# Outputs for economic analyses
In this notebook, we'll use a similar model to the one
introduced in the previous two to think about how we might
assign costs to a state or process within the model 
that we are simulating.
Costs may be associated with either model states
or with transition processes within the model.

We'll again start out by loading some packages,
which you can again ignore unless you're interested.

In [ ]:
%pip install summerepi2==1.3.6
import numpy as np
import pandas as pd
from plotly import express as px
from plotly import graph_objects as go
pd.options.plotting.backend = "plotly"

from summer2 import CompartmentalModel
from summer2.parameters import Parameter

## Capturing the state of being in hospital
In the model in this notebook, we're going to incorporate a new
state to represent people in hospital.
We're going to call this `infectious_hosp` and 
distinguish this from the people who get infected
but don't need to be admitted to hospital
(which we'll call `infectious_nonhosp`).
To do this, we'll split all the new infections to
enter one or other of these two compartments
at the point of infection.
We have done this for simplicity, although
there are significant issues with constructing
the model this way - in particular, that there
is no delay between infection and hospitalisation.
This is clearly not realistic,
and this delay could be included in the model 
relatively easily, but we'll keep it as simple as
possible for now.

In [ ]:
total_population = 1e6  
infectious_seed = 1.0
run_period = [0.0, 50.0]  

# Include both hospitalised and non-hospitalised compartments
model_comps = ["susceptible", "infectious_hosp", "infectious_nonhosp", "recovered"]

# Make both the hospitalised and non-hospitalised compartments infectious
infect_comps = ["infectious_hosp", "infectious_nonhosp"]

# Build the model as previously
hosp_model = CompartmentalModel(times=run_period, compartments=model_comps, infectious_compartments=infect_comps, timestep=0.2)

# Seed the infectious hospitalised compartment only
start_pop = {"susceptible": total_population - infectious_seed, "infectious_hosp": infectious_seed}
hosp_model.set_initial_population(start_pop)

# Split the new infections according to whether they are hospitalised
admission_rate = Parameter("hosp_fraction") * Parameter("contact_rate")
non_admission_rate = (1.0 - Parameter("hosp_fraction")) * Parameter("contact_rate")
hosp_model.add_infection_frequency_flow(name="infection_hosp", contact_rate=admission_rate, source="susceptible", dest="infectious_hosp")
hosp_model.add_infection_frequency_flow(name="infection_nonhosp", contact_rate=non_admission_rate, source="susceptible", dest="infectious_nonhosp")

# Both groups return to the recovered compartment after they're finished in their respective infectious compartment
hosp_model.add_transition_flow(name="recovery_hosp", fractional_rate=Parameter("recovery_rate"), source="infectious_hosp", dest="recovered")
hosp_model.add_transition_flow(name="recovery_nonhosp", fractional_rate=Parameter("recovery_rate"), source="infectious_nonhosp", dest="recovered")

## Incident hospitalisations
Next, let's track the process of hospitalisation, 
rather than the number of people in the hospitalised compartment.
So this is an incident rather than prevalent quantity that 
we can obtain from the simulation.

In [ ]:
hosp_model.request_output_for_flow("infection_hosp", "infection_hosp");

## Outputs
### Prevalent
First let's look at the number of hospitalisations outputs from the model.
As mentioned, we may wish to associated this quantity 
with costs. If we were to do this,
we would want to determine a per-unit time cost,
such as a cost per bed-day for hospital _occupancy_.

In [ ]:
parameters = {
    "contact_rate": 1.5,
    "recovery_rate": 0.2,
    "hosp_fraction": 0.1,
}
hosp_model.run(parameters)
states = hosp_model.get_outputs_df()
states.plot(labels={"index": "time", "value": "number of people"})

### Incident
Modelled incident quantities are relevant to economic simulations,
because there may often be costs associated with transition processes. 
For example, when people are admitted to hospital, 
there may be per-admission costs associated
with the process of being managed through
the emergency department of the hospitals in which the 
patients are being managed.

In [ ]:
derived_outputs = hosp_model.get_derived_outputs_df()
admit_fig = derived_outputs.plot()
admit_fig.update_layout(title="daily new hospital admissions", xaxis={"title": "time"}, yaxis={"title": "new admissions per day"})

## Total costs
Now let's consider how we could build up the total costs of a simulation
from the states and transitions we previously identified.

In [ ]:
occupancy = states["infectious_hosp"]
admissions = derived_outputs["infection_hosp"]

# Cost per hospital bed day
occupancy_cost = 1000.0

# Cost per admission
admission_cost = 500.0

# Costs over time
total_costs_over_time = occupancy * occupancy_cost + admissions * admission_cost

# Total costs over the simulation interval
total_costs_over_period = total_costs_over_time.sum()